# <center>  **<span style="font-size:80px;">Machine Learning</span>** </center>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch
import sys
import os

from pathlib import Path

sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import DatasetKeys, AIConstants, Paths
Paths.init_project()

In [ ]:
from main import WaterApp

from src.water2fraud.models.explainability import ModelExplainer
from src.water2fraud.models.clustering import ClusterManager
from src.water2fraud.models.trainer import plot_reconstruction

In [ ]:
seed = AIConstants.RANDOM_STATE
device = "cuda" if torch.cuda.is_available() else "cpu"

ml_path = Path("./AMAEM/machine_learning")
ml_path.mkdir(parents=True, exist_ok=True)

# Pipeline

## Fase1: Carga y Preprocesamiento

In [ ]:
df_raw = WaterApp._load_data()
X_sequences, metadata_df, feature_names = WaterApp._phase_1_preprocessing(df_raw)

In [ ]:
X_sequences.shape

# 4004: número de muestras
# 12: seq_len = meses
# 9: features = [m3_ratio, mes,                                                                         <- AMAEM
#                ohe_domestico, ohe_comercial, ohe_no_domestico,                                        <- OneHotEncoding
#                num_vt_barrio, porcentaje_vt_barrio, ocupaciones_vt_prov, pernocatciones_vt_prov       <- Turismo
#]

In [ ]:
X_sequences[0][0] # Mes Enero

In [ ]:
X_sequences[0][7] # Mes Agosto

## Fase 2: Clustering

### Búsqueda del Número Óptimo de Clústeres (Método del Codo)

In [ ]:
#mejor_k = ClusterManager.find_optimal_clusters(X_sequences, max_clusters=4, feature_idx=0)
#AIConstants.N_CLUSTERS_DEFAULT = mejor_k

#  > Evaluando k=01 | Inercia: 6.59
#  > Evaluando k=02 | Inercia: 4.92 
#  > Evaluando k=03 | Inercia: 4.27
#  > Evaluando k=04 | Inercia: 3.97
# ...

AIConstants.N_CLUSTERS_DEFAULT

### Time Series K-Means

In [ ]:
labels, cluster_manager = WaterApp._phase_2_clustering(X_sequences)
metadata_df['cluster'] = labels

In [ ]:
ClusterManager.plot_cluster_samples(X_sequences, labels, ml_path, feature_idx=0)

## Fase 3: Autoencoder Training

In [ ]:
# Entrenamos los autoencoders delegando todo a main.py
# Al pasar plot=True, las gráficas aparecerán automáticamente debajo de la celda
modelos_entrenados = WaterApp._phase_3_training(
    X_sequences, 
    metadata_df, 
    device, 
    plot=True
)

## Fase4: Inferencia y Detección

In [ ]:
# Detectamos anomalías cruzando datos con los modelos entrenados
df_ae = WaterApp._phase_4_ae_detection(X_sequences, metadata_df, modelos_entrenados, feature_names, device)

## Fase 5: Risk Scoring (Autoencoder + Físicos)

In [ ]:
df_final = WaterApp._phase_5_risk_scoring(df_ae, risk_score={"ae_weight": 0.8, "physics_weight": 0.2})

sospechosos = df_final[
    (df_final[DatasetKeys.IS_AE_ANOMALY] == True) & 
    (df_final[DatasetKeys.USO] != "COMERCIAL") &
    (df_final[DatasetKeys.USO] != "NO DOMESTICO")
].sort_values(by=DatasetKeys.FRAUD_RISK_SCORE, ascending=False)

print(f"Se han detectado {len(sospechosos)} perfiles de consumo sospechosos de ser viviendas turísticas ilegales.")
display(sospechosos.head(10))

## Inspección Visual

In [ ]:
top_n = 6

if not sospechosos.empty:       
    for i in range(min(top_n, len(sospechosos))):
        idx_seq = sospechosos.index[i]
        cluster_id = sospechosos.loc[idx_seq, DatasetKeys.CLUSTER]

        print(f"Top {i+1} | Índice: {idx_seq} | Clúster: {cluster_id} | "
              f"Barrio: {sospechosos.loc[idx_seq, DatasetKeys.BARRIO]} | "
              f"Error MAE: {sospechosos.loc[idx_seq, 'reconstruction_error']:.4f}")

        secuencia_anomala = X_sequences[idx_seq]
        modelo_evaluador = modelos_entrenados[f"ae_cluster_{cluster_id}"]
        plot_reconstruction(model=modelo_evaluador, sequence=secuencia_anomala, feature_idx=0, feature_name="Ratio Consumo", device=device)
else:
    print("No se encontraron sospechosos")

## Explicabilidad

In [ ]:
feature_names

In [ ]:
indices_anomalos = sospechosos.index

# Extraemos del tensor 3D original solo los que dieron alerta roja
# OJO: X_sequences debe ser un tensor de PyTorch
X_anomalies_tensor = torch.tensor(X_sequences[indices_anomalos], dtype=torch.float32)

# Como "fondo de normalidad", pasamos todo el tensor (o los que dieron False)
X_normal_tensor = torch.tensor(X_sequences, dtype=torch.float32)

In [ ]:
# Lanzamos el Explicador
ModelExplainer.plot_feature_importance(
    model=modelos_entrenados[f"ae_cluster_0"],
    X_train_tensor=X_normal_tensor,
    X_anomalies_tensor=X_anomalies_tensor,
    feature_names=feature_names,
    device="cuda",
    save_path=ml_path / "SHAP_cluster_0.png"
)

In [ ]:
# Lanzamos el Explicador
ModelExplainer.plot_feature_importance(
    model=modelos_entrenados[f"ae_cluster_1"],
    X_train_tensor=X_normal_tensor,
    X_anomalies_tensor=X_anomalies_tensor,
    feature_names=feature_names,
    device="cuda",
    save_path=ml_path / "SHAP_cluster_1.png"
)

# Random Forest & XGBoost

In [ ]:
from src.water2fraud.models.tree_anomaly_validation import TreeModelValidator
X_2d = TreeModelValidator.flatten_sequences(X_sequences)

In [ ]:
# XGBOOST y RANDOM FOREST (Replicadores del Autoencoder)
# Utilizamos el Autoencoder como etiqueta 'y'
y_autoencoder = df_final[DatasetKeys.ALERTA_TURISTICA_ILEGAL].astype(int).values

rf_model, xgb_model, rf_preds, xgb_preds, y_test = TreeModelValidator.run_surrogate_models(X_2d, y_autoencoder)

In [ ]:
# Evaluamos si XGBoost aprendió a pensar igual que el Autoencoder
TreeModelValidator.evaluate_surrogate(y_test, xgb_preds, "XGBoost", ml_path)

In [ ]:
# Evaluamos si Random Forest aprendió a pensar igual que el Autoencoder
TreeModelValidator.evaluate_surrogate(y_test, rf_preds, "Random Forest", ml_path)

## Explicabilidad

In [ ]:
TreeModelValidator.plot_tree_importance(xgb_model, feature_names)

In [ ]:
TreeModelValidator.plot_tree_importance(rf_model, feature_names)